# HW01-A — Dockerized Airbnb Ops Package

Your first job is to make a tiny data package that can run the same way twice.

That sounds boring. Boring reproducibility is how production works.

By the end, you should have:

- a Python package
- a CLI command
- a Docker image
- a Docker Compose service
- a DVC stage
- a processed CSV
- a short run report

## Submission discipline

This is individual work.

Work locally. Push to GitHub. Use the shared server services through URLs and credentials (for homeworks B & C, not this one). Do not SSH into the server.

Do not commit `.env`, `.venv/`, passwords.

## Credentials and shared services

Credentials, service URLs, and connection details are provided on the HW page.

Use those exact values. Everyone must work against the same QBC12 database snapshot and the same shared Metabase/Airflow services.

Do not paste credentials into notebook markdown. Do not commit `.env` files. Do not screenshot passwords.


## Useful references

Use docs when stuck. Guessing is slower.

- Python `pyproject.toml`: https://packaging.python.org/en/latest/guides/writing-pyproject-toml/
- Python CLI entry points: https://setuptools.pypa.io/en/latest/userguide/entry_point.html
- Dockerfile reference: https://docs.docker.com/reference/dockerfile/
- Docker Compose: https://docs.docker.com/compose/
- Docker volumes: https://docs.docker.com/engine/storage/volumes/
- DVC `dvc.yaml`: https://doc.dvc.org/user-guide/project-structure/dvcyaml-files
- DVC `repro`: https://doc.dvc.org/command-reference/repro

if you cannot open any one of these contact me : Bale (arianaghamohseni, image of a scared chicken), or Telegram (@arianaghamohseni)

In [1]:
from pathlib import Path
import textwrap
import pandas as pd

PROJECT = Path.cwd()
for path in ['src/airbnb_ops', 'data/raw', 'data/processed', 'reports', 'tests']:
    (PROJECT / path).mkdir(parents=True, exist_ok=True)
(PROJECT / 'src/airbnb_ops/__init__.py').write_text('__version__ = "0.1.0"\n')
print(PROJECT)

f:\git\MLOps-Assignments\HW01\HW01_A


## 1. Raw inputs

The two cells below are given to you.

Do not change these rows. The point of HW01-A is packaging, Docker, DVC, and validation, not inventing toy data.

Files created:

```text
data/raw/listings_sample.csv
data/raw/neighbourhood_segments.csv
```

The listings file includes `host_name` and `host_id` on purpose. Your PII code must handle them.


In [2]:
# Provided starter data 1.1
# Do not edit this cell.
# Implementation note:
# - This creates the listing-level raw input.
# - The host_name and host_id columns are intentionally included.
# - Later, your PII code must drop host_name and replace host_id with a stable host_key.

listings = pd.DataFrame(
    [
        {
            "listing_id": 1,
            "neighbourhood": "Centrum-West",
            "price": 180,
            "minimum_nights": 2,
            "availability_365": 120,
            "number_of_reviews": 31,
            "host_id": 901,
            "host_name": "Alice",
        },
        {
            "listing_id": 2,
            "neighbourhood": "Centrum-West",
            "price": 210,
            "minimum_nights": 3,
            "availability_365": 80,
            "number_of_reviews": 12,
            "host_id": 902,
            "host_name": "Bob",
        },
        {
            "listing_id": 3,
            "neighbourhood": "De Pijp",
            "price": 135,
            "minimum_nights": 2,
            "availability_365": 210,
            "number_of_reviews": 44,
            "host_id": 903,
            "host_name": "Chris",
        },
        {
            "listing_id": 4,
            "neighbourhood": "Oud-Noord",
            "price": 95,
            "minimum_nights": 1,
            "availability_365": 300,
            "number_of_reviews": 9,
            "host_id": 904,
            "host_name": "Dana",
        },
        {
            "listing_id": 5,
            "neighbourhood": "Oud-Noord",
            "price": 105,
            "minimum_nights": 2,
            "availability_365": 260,
            "number_of_reviews": 18,
            "host_id": 905,
            "host_name": "Eve",
        },
        {
            "listing_id": 6,
            "neighbourhood": "De Baarsjes",
            "price": 125,
            "minimum_nights": 3,
            "availability_365": 170,
            "number_of_reviews": 27,
            "host_id": 906,
            "host_name": "Farid",
        },
        {
            "listing_id": 7,
            "neighbourhood": "De Baarsjes",
            "price": 145,
            "minimum_nights": 4,
            "availability_365": 90,
            "number_of_reviews": 21,
            "host_id": 907,
            "host_name": "Grace",
        },
        {
            "listing_id": 8,
            "neighbourhood": "Westerpark",
            "price": 155,
            "minimum_nights": 2,
            "availability_365": 140,
            "number_of_reviews": 36,
            "host_id": 908,
            "host_name": "Hamed",
        },
    ]
)

listings.to_csv(PROJECT / "data/raw/listings_sample.csv", index=False)
listings.head()


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


In [3]:
# Provided starter data 1.2
# Do not edit this cell.
# Implementation note:
# - This file enriches neighbourhoods with business metadata.
# - Your transform step should join this file after aggregation.
# - If a neighbourhood has no segment row, your code should fill it with "unknown".

segments = pd.DataFrame(
    [
        {
            "neighbourhood": "Centrum-West",
            "tourism_segment": "tourist-heavy",
            "priority_level": "high",
        },
        {
            "neighbourhood": "De Pijp",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "Oud-Noord",
            "tourism_segment": "emerging",
            "priority_level": "medium",
        },
        {
            "neighbourhood": "De Baarsjes",
            "tourism_segment": "local-heavy",
            "priority_level": "low",
        },
        {
            "neighbourhood": "Westerpark",
            "tourism_segment": "mixed",
            "priority_level": "medium",
        },
    ]
)

segments.to_csv(PROJECT / "data/raw/neighbourhood_segments.csv", index=False)
segments


,neighbourhood,tourism_segment,priority_level
0,Centrum-West,tourist-heavy,high
1,De Pijp,mixed,medium
2,Oud-Noord,emerging,medium
3,De Baarsjes,local-heavy,low
4,Westerpark,mixed,medium


In [4]:
# Checkpoint
for file in ['data/raw/listings_sample.csv', 'data/raw/neighbourhood_segments.csv']:
    assert Path(file).exists(), f'Missing {file}'

pd.read_csv('data/raw/listings_sample.csv').head()

,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_id,host_name
0,1,Centrum-West,180,2,120,31,901,Alice
1,2,Centrum-West,210,3,80,12,902,Bob
2,3,De Pijp,135,2,210,44,903,Chris
3,4,Oud-Noord,95,1,300,9,904,Dana
4,5,Oud-Noord,105,2,260,18,905,Eve


## 2. Data contract

Write code against a contract, not vibes.

Input listing columns:

```text
listing_id, neighbourhood, price, minimum_nights,
availability_365, number_of_reviews, host_id, host_name
```

Output columns:

```text
neighbourhood, num_listings, avg_price, median_price,
avg_minimum_nights, availability_365_avg, total_reviews,
reviews_per_listing, tourism_segment, priority_level
```

In [ ]:
# TODO 2.1
# Create src/airbnb_ops/config.py.
# Add a PipelineConfig dataclass with default paths for:
# listings_path, segments_path, output_path, report_path

# Write your code here.
# Verify config.py was created
from src.airbnb_ops.config import PipelineConfig
cfg = PipelineConfig()
print(cfg)

# config.py code
# ! Not for running
from dataclasses import dataclass, field
from pathlib import Path

# This is the src/airbnb_ops/ directory
_PACKAGE_DIR = Path(__file__).parent

# Walk up to the project root (src/airbnb_ops -> src -> project root)
_PROJECT_ROOT = _PACKAGE_DIR.parent.parent

@dataclass
class PipelineConfig:
    listings_path: Path = field(default_factory=lambda: _PROJECT_ROOT / "data/raw/listings_sample.csv")
    segments_path: Path = field(default_factory=lambda: _PROJECT_ROOT / "data/raw/neighbourhood_segments.csv")
    output_path: Path = field(default_factory=lambda: _PROJECT_ROOT / "data/processed/airbnb_neighbourhood_summary.csv")
    report_path: Path = field(default_factory=lambda: _PROJECT_ROOT / "reports/hw01_a_run_report.md")

PipelineConfig(listings_path=WindowsPath('f:/git/MLOps-Assignments/HW01/HW01_A/data/raw/listings_sample.csv'), segments_path=WindowsPath('f:/git/MLOps-Assignments/HW01/HW01_A/data/raw/neighbourhood_segments.csv'), output_path=WindowsPath('f:/git/MLOps-Assignments/HW01/HW01_A/data/processed/airbnb_neighbourhood_summary.csv'), report_path=WindowsPath('f:/git/MLOps-Assignments/HW01/HW01_A/reports/hw01_a_run_report.md'))


In [ ]:
# TODO 2.2
# Create src/airbnb_ops/extract.py.
# Add read_csv_checked(path: Path) -> pd.DataFrame.
# It should raise FileNotFoundError if the file is missing.

# Write your code here.
from src.airbnb_ops.extract import read_csv_checked
listing_df = read_csv_checked("data/raw/listings_sample.csv")
segment_df = read_csv_checked("data/raw/neighbourhood_segments.csv")
# extract.py code
# ! Not for running
from pathlib import Path
import pandas as pd

def read_csv_checked(path: Path) -> pd.DataFrame:
    """Read a CSV file, raising FileNotFoundError if it doesn't exist."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")
    return pd.read_csv(path)

## 3. PII handling

For this homework:

- drop `host_name`
- convert `host_id` to `host_key`
- drop the original `host_id`

Do not use Python's built-in `hash()`. It is not stable across sessions. Use `hashlib.sha256`.

In [ ]:
# TODO 3.1
# Create src/airbnb_ops/pii.py.
# Add:
# - DIRECT_PII_COLUMNS
# - pseudonymize_value(value, salt='qbc12')
# - handle_pii(df)

# Write your code here.
from src.airbnb_ops.pii import handle_pii

clean = handle_pii(listing_df)
print(clean.columns.tolist())
assert "host_name" not in clean.columns
assert "host_id" not in clean.columns
assert "host_key" in clean.columns
clean.head()

# pii.py code
# ! Not for running
import hashlib
import pandas as pd

# Columns to drop outright
DIRECT_PII_COLUMNS = ["host_name"]

def pseudonymize_value(value, salt: str = "qbc12") -> str:
    """Hash a value with a salt using SHA-256 for stable pseudonymization."""
    raw = f"{salt}:{value}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def handle_pii(df: pd.DataFrame) -> pd.DataFrame:
    """Drop PII columns and pseudonymize host_id -> host_key."""
    df = df.copy()
    
    # Drop direct PII columns that exist
    cols_to_drop = [c for c in DIRECT_PII_COLUMNS if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    
    # Pseudonymize host_id into host_key, then drop host_id
    if "host_id" in df.columns:
        df["host_key"] = df["host_id"].apply(pseudonymize_value)
        df = df.drop(columns=["host_id"])
    
    return df

['listing_id', 'neighbourhood', 'price', 'minimum_nights', 'availability_365', 'number_of_reviews', 'host_key']


,listing_id,neighbourhood,price,minimum_nights,availability_365,number_of_reviews,host_key
0,1,Centrum-West,180,2,120,31,157b469a56bea3d8d61f8289988c48277d41246ee70edd...
1,2,Centrum-West,210,3,80,12,21abc69946417e590e593249bb58a1ebcf5a2c936e528d...
2,3,De Pijp,135,2,210,44,787c7dcd2e488c99341b4db880069c8db64705a280e69f...
3,4,Oud-Noord,95,1,300,9,dc29735cbec8fb0500ef68b24c729185a1beaf51ed4e2d...
4,5,Oud-Noord,105,2,260,18,29bbea2a5bd02fb08e5c4bd9c03c13414d1cfa2de8d0c8...


## 4. Transform

Build a neighbourhood-level summary. Use `groupby`, not loops.

Join the segment metadata after aggregation. If a segment is missing, fill it with `unknown`.

In [ ]:
# TODO 4.1
# Create src/airbnb_ops/transform.py.
# Add build_neighbourhood_summary(listings, segments).
# It should validate required input columns, aggregate, join segments, and return a dataframe.

# Write your code here.
from src.airbnb_ops.transform import build_neighbourhood_summary

summary = build_neighbourhood_summary(clean, segment_df)
print(summary.columns.tolist())
summary

# transform.py code
# ! Not for running
import pandas as pd

REQUIRED_INPUT_COLUMNS = {
    "listing_id", "neighbourhood", "price", "minimum_nights",
    "availability_365", "number_of_reviews"
}

def build_neighbourhood_summary(listings: pd.DataFrame, segments: pd.DataFrame) -> pd.DataFrame:
    """Aggregate listings by neighbourhood and join segment metadata."""
    # Validate required columns
    missing = REQUIRED_INPUT_COLUMNS - set(listings.columns)
    if missing:
        raise ValueError(f"Missing required columns in listings: {missing}")

    # Aggregate by neighbourhood
    summary = listings.groupby("neighbourhood").agg(
        num_listings=("listing_id", "count"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        avg_minimum_nights=("minimum_nights", "mean"),
        availability_365_avg=("availability_365", "mean"),
        total_reviews=("number_of_reviews", "sum"),
    ).reset_index()

    # Derived metric
    summary["reviews_per_listing"] = summary["total_reviews"] / summary["num_listings"]

    # Left join segments so neighbourhoods without a segment get NaN → "unknown"
    summary = summary.merge(segments[["neighbourhood", "tourism_segment", "priority_level"]],
                            on="neighbourhood", how="left")
    summary["tourism_segment"] = summary["tourism_segment"].fillna("unknown")
    summary["priority_level"] = summary["priority_level"].fillna("unknown")

    return summary

['neighbourhood', 'num_listings', 'avg_price', 'median_price', 'avg_minimum_nights', 'availability_365_avg', 'total_reviews', 'reviews_per_listing', 'tourism_segment', 'priority_level']


,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## 5. Validation

Validation is the bouncer. Bad output does not get into the club.

Minimum checks:

- output is not empty
- required output columns exist
- no PII columns exist
- neighbourhood is not null
- num_listings > 0
- avg_price >= 0
- availability_365_avg between 0 and 365

In [ ]:
# TODO 5.1
# Create src/airbnb_ops/validate.py.
# Add validate_summary(summary) -> None.
# Raise ValueError when a check fails.

# Write your code here.
from src.airbnb_ops.validate import validate_summary
validate_summary(summary)  # should not raise
print("Validation passed!")

# validate.py code
# ! Not for running
import pandas as pd

REQUIRED_OUTPUT_COLUMNS = {
    "neighbourhood", "num_listings", "avg_price", "median_price",
    "avg_minimum_nights", "availability_365_avg", "total_reviews",
    "reviews_per_listing", "tourism_segment", "priority_level"
}

PII_COLUMNS = {"host_name", "host_id", "reviewer_name", "reviewer_id",
               "listing_url", "host_url"}

def validate_summary(summary: pd.DataFrame) -> None:
    """Validate the neighbourhood summary dataframe. Raises ValueError on failure."""
    if summary.empty:
        raise ValueError("Output dataframe is empty.")

    missing_cols = REQUIRED_OUTPUT_COLUMNS - set(summary.columns)
    if missing_cols:
        raise ValueError(f"Missing required output columns: {missing_cols}")

    leaked_pii = PII_COLUMNS & set(summary.columns)
    if leaked_pii:
        raise ValueError(f"PII columns found in output: {leaked_pii}")

    if summary["neighbourhood"].isnull().any():
        raise ValueError("Null values found in 'neighbourhood' column.")

    if (summary["num_listings"] <= 0).any():
        raise ValueError("'num_listings' must be > 0 for all rows.")

    if (summary["avg_price"] < 0).any():
        raise ValueError("'avg_price' must be >= 0 for all rows.")

    if not summary["availability_365_avg"].between(0, 365).all():
        raise ValueError("'availability_365_avg' must be between 0 and 365.")

Validation passed!


## 6. CLI

The package should expose one command:

```bash
airbnb-ops run
```

The command should read inputs, handle PII, transform, validate, write CSV, and write a markdown report.

In [ ]:
# TODO 6.1
# Create src/airbnb_ops/cli.py using Typer.
# Add a run command.
# It should write:
# - data/processed/airbnb_neighbourhood_summary.csv
# - reports/hw01_a_run_report.md

# Write your code here.
!python -m airbnb_ops.cli run

# cli.py code
# ! Not for running
import typer
from pathlib import Path
from datetime import datetime

from airbnb_ops.config import PipelineConfig
from airbnb_ops.extract import read_csv_checked
from airbnb_ops.pii import handle_pii
from airbnb_ops.transform import build_neighbourhood_summary
from airbnb_ops.validate import validate_summary

app = typer.Typer()

@app.command("run")
def run():
    """Run the full Airbnb neighbourhood summary pipeline."""
    cfg = PipelineConfig()

    typer.echo("Reading raw data...")
    listings = read_csv_checked(cfg.listings_path)
    segments = read_csv_checked(cfg.segments_path)

    typer.echo("Handling PII...")
    listings = handle_pii(listings)

    typer.echo("Transforming data...")
    summary = build_neighbourhood_summary(listings, segments)

    typer.echo("Validating output...")
    validate_summary(summary)

    typer.echo("Writing output CSV...")
    cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(cfg.output_path, index=False)

    typer.echo("Writing report...")
    cfg.report_path.parent.mkdir(parents=True, exist_ok=True)
    report = f"""# HW01-A Run Report

            Generated: {datetime.now().isoformat()} UTC

            ## Summary

            - Neighbourhoods processed: {len(summary)}
            - Total listings: {summary['num_listings'].sum()}
            - Average price across all neighbourhoods: {summary['avg_price'].mean():.2f}

            ## Output

            Saved to: `{cfg.output_path}`

            ## Validation

            All checks passed!
        """
    cfg.report_path.write_text(report)
    typer.echo(f"Done. Output: {cfg.output_path}")
    
@app.command()
def version():
    """Print the package version."""
    from airbnb_ops import __version__
    typer.echo(__version__)


## 7. Package metadata

`pyproject.toml` makes the project installable. `[project.scripts]` creates the `airbnb-ops` command.

In [ ]:
# TODO 7.1
# Create pyproject.toml.
# Requirements:
# - project name: airbnb-ops
# - dependencies: pandas, typer, rich
# - script: airbnb-ops = airbnb_ops.cli:app

# Write your code here.
# pyproject.toml code
# ! Not for running
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "airbnb-ops"
version = "0.1.0"
requires-python = ">=3.10"
dependencies = [
    "pandas",
    "typer",
    "rich",
]

[project.scripts]
airbnb-ops = "airbnb_ops.cli:app"

[tool.setuptools.packages.find]
where = ["src"]

In [ ]:
# TODO 7.2
# Create requirements.txt with pandas, typer, rich, dvc.

# Write your code here.
# requirements.txt code
# ! Not for running
pandas
typer
rich
dvc

In [17]:
# Local smoke test
import sys
import pandas as pd

!{sys.executable} -m pip install -q -e .
!airbnb-ops run
pd.read_csv('data/processed/airbnb_neighbourhood_summary.csv').head()

Reading raw data...
Handling PII...
Transforming data...
Validating output...
Writing output CSV...
Writing report...
Done. Output: F:\git\MLOps-Assignments\HW01\HW01_A\data\processed\airbnb_neighbourhood_summary.csv


,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## 8. Docker

Create `.dockerignore`, `Dockerfile`, and `docker-compose.yml`.

Watch out for this common garbage move: copying generated outputs into the image. Do not do that. Mount `data/` and `reports/` so the container writes outputs to your working directory.

In [ ]:
# TODO 8.1
# Create .dockerignore.
# Exclude .git, .venv, __pycache__, .ipynb_checkpoints, data/processed, reports.

# Write your code here.
# .dockerignore code
# ! Not for running
.git
.venv
__pycache__
.ipynb_checkpoints
data/processed
reports
*.pyc
*.egg-info

In [ ]:
# TODO 8.2
# Create Dockerfile.
# Requirements:
# - FROM python:3.11-slim
# - WORKDIR /app
# - copy package metadata and src/
# - install requirements and package
# - default command: airbnb-ops run

# Write your code here.
# .dockerignore code
# ! Not for running
FROM python:3.11-slim

WORKDIR /app

# Copy only what's needed to install the package
COPY requirements.txt .
COPY pyproject.toml .
COPY src/ src/

# Install dependencies and the package
RUN pip install --no-cache-dir -r requirements.txt && \
    pip install --no-cache-dir -e .

# Default command
CMD ["airbnb-ops", "run"]

In [ ]:
# TODO 8.3
# Create docker-compose.yml.
# Requirements:
# - service name: airbnb-ops
# - build current directory
# - mount ./data and ./reports
# - command: airbnb-ops run

# Write your code here.
# docker-compose.yml code
# ! Not for running
services:
  airbnb-ops:
    build: .
    volumes:
      - ./data:/app/data
      - ./reports:/app/reports
    command: airbnb-ops run

In [18]:
# Docker smoke test. Run in terminal if notebook cannot access Docker.
!docker compose build
!docker compose run --rm airbnb-ops

#1 [internal] load local bake definitions

 Image hw01_a-airbnb-ops Building 
 Image hw01_a-airbnb-ops Built 



#1 reading from stdin 547B done
#1 DONE 0.0s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile:
#2 transferring dockerfile: 376B 0.2s done
#2 DONE 0.5s

#3 [internal] load metadata for docker.io/library/python:3.10-slim
#3 DONE 3.2s

#4 [internal] load .dockerignore
#4 transferring context: 128B 0.0s done
#4 DONE 0.2s

#5 [internal] load build context
#5 DONE 0.0s

#6 [1/6] FROM docker.io/library/python:3.10-slim@sha256:70f65c721aaddfb22b20ed6ec12606c59d9592493c5fcb6639f3d0e8ba3fbc10
#6 resolve docker.io/library/python:3.10-slim@sha256:70f65c721aaddfb22b20ed6ec12606c59d9592493c5fcb6639f3d0e8ba3fbc10
#6 resolve docker.io/library/python:3.10-slim@sha256:70f65c721aaddfb22b20ed6ec12606c59d9592493c5fcb6639f3d0e8ba3fbc10 0.4s done
#6 DONE 0.4s

#5 [internal] load build context
#5 transferring context: 2.28kB 0.0s done
#5 DONE 0.3s

#7 [2/6] WORKDIR /app
#7 CACHED

#8 [3/6] COPY requirements.txt .
#8 CACHED

#9 [4/6] COPY pyproject.toml .
#9 CACHED

#10 [5/6] CO

 Container hw01_a-airbnb-ops-run-ed4f62520a0f Creating 
 Container hw01_a-airbnb-ops-run-ed4f62520a0f Created 


## 9. DVC

A DVC stage is the receipt for how your output was produced.

Create one stage named `run_pipeline`.

In [ ]:
# TODO 9.1
# Create dvc.yaml.
# Stage:
# - cmd: airbnb-ops run
# - deps: src/airbnb_ops, both raw CSV files
# - outs: processed CSV
# - metrics/report: reports/hw01_a_run_report.md with cache: false

# Write your code here.
# dvc.yaml code
# ! Not for running
stages:
  run_pipeline:
    cmd: airbnb-ops run
    deps:
      - src/airbnb_ops
      - data/raw/listings_sample.csv
      - data/raw/neighbourhood_segments.csv
    outs:
      - data/processed/airbnb_neighbourhood_summary.csv
    metrics:
      - reports/hw01_a_run_report.md:
          cache: false

In [20]:
# Extra credit 9.2
# Run these commands if DVC is installed in your local environment.
# This is a useful check, but the required part is creating a correct dvc.yaml file.

!dvc repro
!dvc dag


Running stage 'run_pipeline':
> airbnb-ops run
Reading raw data...
Handling PII...
Transforming data...
Validating output...
Writing output CSV...
Writing report...
Done. Output: F:\git\MLOps-Assignments\HW01\HW01_A\data\processed\airbnb_neighbourhood_summary.csv
Generating lock file 'dvc.lock'
Updating lock file 'dvc.lock'

To track the changes with git, run:

	git add dvc.lock 'data\processed\.gitignore'

To enable auto staging, run:

	dvc config core.autostage true
Use `dvc push` to send your updates to remote storage.
+--------------+ 
| run_pipeline | 
+--------------+ 
+------------------------------------+ 
| ..\..\HW01_A\dvc.yaml:run_pipeline | 
+------------------------------------+ 


## Final proof

If this cell fails, HW01-A is not done.

In [21]:
output = Path('data/processed/airbnb_neighbourhood_summary.csv')
report = Path('reports/hw01_a_run_report.md')
assert output.exists(), 'Output file was not created.'
assert report.exists(), 'Report file was not created.'

df = pd.read_csv(output)
required = {'neighbourhood','num_listings','avg_price','median_price','avg_minimum_nights','availability_365_avg','total_reviews','reviews_per_listing','tourism_segment','priority_level'}
assert not (required - set(df.columns)), f'Missing: {required - set(df.columns)}'
for bad in ['host_name','host_id','reviewer_name','reviewer_id','listing_url','host_url']:
    assert bad not in df.columns, f'PII leaked: {bad}'
df.head()

,neighbourhood,num_listings,avg_price,median_price,avg_minimum_nights,availability_365_avg,total_reviews,reviews_per_listing,tourism_segment,priority_level
0,Centrum-West,2,195.0,195.0,2.5,100.0,43,21.5,tourist-heavy,high
1,De Baarsjes,2,135.0,135.0,3.5,130.0,48,24.0,local-heavy,low
2,De Pijp,1,135.0,135.0,2.0,210.0,44,44.0,mixed,medium
3,Oud-Noord,2,100.0,100.0,1.5,280.0,27,13.5,emerging,medium
4,Westerpark,1,155.0,155.0,2.0,140.0,36,36.0,mixed,medium


## Commit

```bash
git add .
git commit -m "HW01-A dockerized Airbnb package"
```